In [1]:
# ============================================================
# Phase 0 — Dataset Inspection
# No modeling. No cleaning decisions yet. Only facts.
# ============================================================
import pandas as pd
import numpy as np
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

from pathlib import Path

DATA_PATH = "data/raw/european_household_energy.csv"

DATA_PATH = Path("../data/raw/european_household_energy.csv")

if not DATA_PATH.exists():
    DATA_PATH = Path("data/raw/european_household_energy.csv")

if not DATA_PATH.exists():
    candidates = [DATA_PATH, RAW_PATH]
    candidates += [
        parent / "data" / "raw" / "european_household_energy.csv"
        for parent in (Path.cwd(), *Path.cwd().parents)
    ]

    # Use the first candidate that exists.
    DATA_PATH = next(
        (path for path in candidates if path.exists()),
        None,
    )

    if DATA_PATH is None:
        tried = "\n- ".join(str(path.resolve()) for path in candidates)
        search_roots = [Path.cwd(), Path.home() / "Downloads"]
        found = next(
            (
            path
            for root in search_roots
            if root.exists()
            for path in root.rglob("european_household_energy.csv")
            if path.is_file()
            ),
            None,
        )

        if found is not None:
            DATA_PATH = found
        else:
            raise FileNotFoundError(
            f"CSV not found. Copy european_household_energy.csv into "
            f"{Path.cwd() / 'data' / 'raw'} or set DATA_PATH to its full path.\n"
            f"Paths checked:\n- {tried}"
            )

df = pd.read_csv(DATA_PATH)
"Loaded ✅"

'Loaded ✅'

In [2]:
pd.DataFrame({
    "rows": [len(df)],
    "columns": [df.shape[1]],
    "column_names": [", ".join(df.columns)],
})

,rows,columns,column_names
0,33874,14,"country_code, country_iso2, country, is_countr..."


In [3]:
dtypes_df = pd.DataFrame({
    "column": df.columns,
    "dtype": df.dtypes.astype(str).values,
    "n_unique": [df[c].nunique() for c in df.columns],
    "n_null": [df[c].isna().sum() for c in df.columns],
    "null_pct": [(df[c].isna().mean() * 100).round(2) for c in df.columns],
})

dtypes_df

,column,dtype,n_unique,n_null,null_pct
0,country_code,str,40,0,0.00
1,country_iso2,str,39,518,1.53
2,country,str,40,0,0.00
3,is_country_group,bool,2,0,0.00
4,end_use_code,str,7,0,0.00
5,end_use,str,7,0,0.00
6,fuel_code,str,14,0,0.00
7,fuel,str,14,0,0.00
8,period,int64,15,0,0.00
9,date,str,15,0,0.00


In [4]:
groups_vs_countries = (
    df.groupby("is_country_group")
    .agg(
        rows=("country", "size"),
        unique_countries=("country", "nunique"),
    )
    .assign(pct_of_rows=lambda x: (x["rows"] / x["rows"].sum() * 100).round(2))
)

groups_vs_countries

,rows,unique_countries,pct_of_rows
is_country_group,,,
False,33356,39,98.47
True,518,1,1.53


In [5]:
# List of real countries (exclude aggregates)
real_countries = (
    df.loc[df.is_country_group == False, "country"]
    .drop_duplicates().sort_values().reset_index(drop=True)
)

pd.DataFrame({"real_countries": real_countries, "n": len(real_countries)}).T

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38
real_countries,Albania,Austria,Belgium,Bosnia and Herzegovina,Bulgaria,Croatia,Cyprus,Czechia,Denmark,Estonia,Finland,France,Georgia,Germany,Greece,Hungary,Ireland,Italy,Kosovo*,Latvia,Lithuania,Luxembourg,Malta,Moldova,Montenegro,Netherlands,North Macedonia,Norway,Poland,Portugal,Romania,Serbia,Slovakia,Slovenia,Spain,Sweden,Turkiye,Ukraine,United Kingdom
n,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39,39


In [6]:
years_overview = pd.DataFrame({
    "min_year": [df.year.min()],
    "max_year": [df.year.max()],
    "n_years": [df.year.nunique()],
    "years": [sorted(df.year.unique().tolist())],
})

years_overview

,min_year,max_year,n_years,years
0,2010,2024,15,"[2010, 2011, 2012, 2013, 2014, 2015, 2016, 201..."


In [7]:
# Coverage per country (how many years each country has)
coverage = (
    df[df.is_country_group == False]
    .groupby("country")["year"].nunique()
    .sort_values()
    .to_frame(name="n_years")
)

coverage["status"] = np.where(coverage["n_years"] == df.year.nunique(), "full", "partial")

pd.DataFrame({
    "full_coverage_countries": [(coverage.status == "full").sum()],
    "partial_coverage_countries": [(coverage.status == "partial").sum()],
    "min_years": [coverage.n_years.min()],
    "max_years": [coverage.n_years.max()],
})

,full_coverage_countries,partial_coverage_countries,min_years,max_years
0,17,22,3,15


In [8]:
enduse_overview = (
    df[["end_use_code", "end_use"]]
    .drop_duplicates()
    .sort_values("end_use_code")
    .reset_index(drop=True)
)

enduse_overview

,end_use_code,end_use
0,FC_OTH_HH_E,Final consumption - other sectors - households...
1,FC_OTH_HH_E_CK,Final consumption - other sectors - households...
2,FC_OTH_HH_E_LE,Final consumption - other sectors - households...
3,FC_OTH_HH_E_OE,Final consumption - other sectors - households...
4,FC_OTH_HH_E_SC,Final consumption - other sectors - households...
5,FC_OTH_HH_E_SH,Final consumption - other sectors - households...
6,FC_OTH_HH_E_WH,Final consumption - other sectors - households...


In [9]:
fuel_overview = (
    df[["fuel_code", "fuel"]]
    .drop_duplicates()
    .sort_values("fuel_code")
    .reset_index(drop=True)
)

fuel_overview

,fuel_code,fuel
0,E7000,Electricity
1,G3000,Natural gas
2,H8000,Heat
3,O4000,Oil and petroleum products
4,O4630,Liquefied petroleum gases
5,O4669,Other kerosene
6,O4671,Gas oil and diesel oil
7,R5110-5150_W6000RI,Primary solid biofuels
8,R5300,Biogases
9,RA000,Renewables and biofuels


In [10]:
target_checks = pd.DataFrame({
    "check": [
        "negative_values",
        "zero_values",
        "positive_values",
        "total_rows",
        "min_positive",
        "max",
        "mean_positive",
        "median_positive",
        "skewness",
    ],
    "value": [
        int((df.energy_tj < 0).sum()),
        int((df.energy_tj == 0).sum()),
        int((df.energy_tj > 0).sum()),
        len(df),
        float(df.loc[df.energy_tj > 0, "energy_tj"].min()),
        float(df.energy_tj.max()),
        float(df.loc[df.energy_tj > 0, "energy_tj"].mean()),
        float(df.loc[df.energy_tj > 0, "energy_tj"].median()),
        float(df.loc[df.energy_tj > 0, "energy_tj"].skew()),
    ],
})

target_checks

,check,value
0,negative_values,0.00
1,zero_values,"14,742.00"
2,positive_values,"19,132.00"
3,total_rows,"33,874.00"
4,min_positive,0.00
5,max,"10,975,898.71"
6,mean_positive,"56,633.50"
7,median_positive,"3,962.79"
8,skewness,20.08


In [11]:
key_cols = ["country_code", "year", "end_use_code", "fuel_code"]

dup_checks = pd.DataFrame({
    "check": [
        "full_duplicate_rows",
        "key_duplicate_rows",
    ],
    "count": [
        int(df.duplicated().sum()),
        int(df.duplicated(subset=key_cols).sum()),
    ],
})

dup_checks

,check,count
0,full_duplicate_rows,0
1,key_duplicate_rows,0


In [12]:
# For each country, are the years continuous?
# Continuity check for TOTAL household energy only.
total_key = df[
    (df.is_country_group == False)
    & (df.end_use_code == "FC_OTH_HH_E")
    & (df.fuel_code == "TOTAL")
]

continuity = (
    total_key
    .groupby("country")["year"]
    .agg(min_year="min", max_year="max", n_years="nunique")
)
continuity["expected_years"] = continuity.max_year - continuity.min_year + 1
continuity["is_continuous"] = continuity.n_years == continuity.expected_years

pd.DataFrame({
    "continuous_countries": [int(continuity.is_continuous.sum())],
    "non_continuous_countries": [int((~continuity.is_continuous).sum())],
    "countries_with_full_history": [int((continuity.n_years == df.year.nunique()).sum())],
})

,continuous_countries,non_continuous_countries,countries_with_full_history
0,38,1,17


In [13]:
flags_summary = (
    df.assign(flag_value=df.flag.fillna("(none)"))
    .groupby("flag_value")
    .agg(rows=("flag", "size"), meaning=("flag_meaning", "first"))
    .reset_index()
    .sort_values("rows", ascending=False)
)

flags_summary

,flag_value,rows,meaning
0,(none),33803,NaN
1,e,70,estimated
2,u,1,low reliability


In [14]:
# Consolidated Phase 0 findings
observations = pd.DataFrame({
    "observation": [
        "Total rows",
        "Total columns",
        "Real countries",
        "Country groups (aggregates)",
        "Years available",
        "Countries with FULL year coverage",
        "Countries with PARTIAL coverage",
        "Negative target values",
        "Zero target values",
        "Target skewness (positive only)",
        "Key-level duplicates",
        "Continuous TOTAL series per country",
    ],
    "value": [
        len(df),
        df.shape[1],
        int(df.loc[df.is_country_group == False, "country"].nunique()),
        int(df.loc[df.is_country_group == True, "country"].nunique()),
        f"{df.year.min()}–{df.year.max()} ({df.year.nunique()} yrs)",
        int((coverage.status == "full").sum()),
        int((coverage.status == "partial").sum()),
        int((df.energy_tj < 0).sum()),
        int((df.energy_tj == 0).sum()),
        round(float(df.loc[df.energy_tj > 0, "energy_tj"].skew()), 2),
        int(df.duplicated(subset=key_cols).sum()),
        int(continuity.is_continuous.sum()),
    ],
})

observations

,observation,value
0,Total rows,33874
1,Total columns,14
2,Real countries,39
3,Country groups (aggregates),1
4,Years available,2010–2024 (15 yrs)
5,Countries with FULL year coverage,17
6,Countries with PARTIAL coverage,22
7,Negative target values,0
8,Zero target values,14742
9,Target skewness (positive only),20.08


# 🗺️ Final Modeling Roadmap — Based on Actual Inspection

> This roadmap is derived from the Phase 0 results — **not** from assumptions.

## Phase 1 — `01_Data_Understanding.ipynb`
- Full column dictionary (English).
- Coverage heatmap per country.
- Target distribution + log-scale view.
- Document every data-quality issue in a table.
- **Decision gate:** which countries are eligible for which module.

## Phase 2 — `02_Data_Preparation.ipynb`
- Filter real countries only (`is_country_group == False`).
- Tag fuel hierarchy: `aggregate / primary / sub_fuel`.
- Classify zeros: `valid_zero` vs `all_zero_series`.
- Build three analytical views:
  - `panel_level`: (country, year, end_use, fuel) — for prediction.
  - `country_total`: (country, year) with TOTAL only — for forecasting.
  - `country_profile`: static per-country summary — for clustering.
- Save to `data/processed/`.

## Phase 3 — `03_Energy_Prediction.ipynb`
**Observation level:** country × year × end_use × fuel (panel).
**Target:** `energy_tj` (log1p-transformed given skewness).
**Split:** chronological — Train ≤2019, Val 2020–2022, Test 2023–2024 *(pending Phase 0 confirmation that all splits have enough rows)*.
**Features:** lag_1, lag_2, lag_3, rolling_3, rolling_5, growth_rate, country_code, end_use, fuel.
**Models:** Dummy, Linear, Ridge, RandomForest, ExtraTrees, HistGradientBoosting.
**Metrics:** MAE, RMSE, R², WAPE.
**Explainability:** Permutation Importance + built-in importance.

## Phase 4 — `04_Energy_Forecasting.ipynb`
**Series:** `country_total` only (TOTAL household energy per country-year).
**Eligible countries:** continuous series + ≥10 years coverage *(Phase 0 will confirm exact count)*.
**Models:** Naive, Drift, Linear Trend, SES, Holt, ARIMA(p,d,q) selected by AIC.
**Backtest:** expanding-window, 3 origins.
**Forecast horizon:** 2025, 2026, 2027 with 80%/95% intervals.

## Phase 5 — `05_Country_Clustering.ipynb`
**Data:** one row per country, ~8 features (avg energy, variability, growth, fuel shares, end-use shares).
**Scaling:** StandardScaler (RobustScaler as sensitivity check).
**Models:** KMeans + Agglomerative.
**k selection:** Elbow + Silhouette.
**Interpretation:** cluster profiles table + PCA 2D map.

## Phase 6 — `06_Final_ML_Analysis.ipynb`
- Model comparison summary.
- Feature importance synthesis.
- Forecast summary table.
- Cluster assignment table.
- Business recommendations.
- README + requirements.txt.

## Guardrails (enforced everywhere)
- ❌ No aggregate + component mixing.
- ❌ No random splits for time data.
- ❌ No future information in features.
- ❌ No fabricated metrics.
- ✅ Reproducible (`random_state=42`).
- ✅ Saved models + preprocessing objects.

In [15]:
# ============================================================
# Phase 0.5 — Pre-Phase-1 sanity checks
# Validate end-use x fuel availability and zero structure
# ============================================================

# --- Check 1: End-use × Fuel combinations actually present ---
combo_matrix = (
    df[df.is_country_group == False]
    .groupby(["end_use_code", "fuel_code"])
    .size()
    .unstack(fill_value=0)
    .astype(int)
)

combo_summary = pd.DataFrame({
    "metric": [
        "total_possible_combos",
        "combos_with_data",
        "combos_missing",
    ],
    "value": [
        combo_matrix.size,
        (combo_matrix > 0).sum().sum(),
        (combo_matrix == 0).sum().sum(),
    ],
})

combo_summary

,metric,value
0,total_possible_combos,98
1,combos_with_data,74
2,combos_missing,24


In [16]:
# --- Check 2: Zero ratio per fuel (structural zeros detection) ---
zero_by_fuel = (
    df[df.is_country_group == False]
    .groupby("fuel")
    .agg(
        rows=("energy_tj", "size"),
        zeros=("energy_tj", lambda s: (s == 0).sum()),
        positives=("energy_tj", lambda s: (s > 0).sum()),
    )
    .assign(zero_pct=lambda x: (x["zeros"] / x["rows"] * 100).round(1))
    .sort_values("zero_pct", ascending=False)
)

zero_by_fuel

,rows,zeros,positives,zero_pct
fuel,,,,
Biogases,2235,2129,106,95.30
Other kerosene,2235,1912,323,85.50
Ambient heat (heat pumps),2235,1557,678,69.70
Solar thermal,2235,1398,837,62.60
Gas oil and diesel oil,2235,1305,930,58.40
Heat,2256,1264,992,56.00
"Solid fossil fuels, peat, peat products, oil shale and oil sands",2260,1146,1114,50.70
Liquefied petroleum gases,2259,798,1461,35.30
Natural gas,2263,731,1532,32.30


In [17]:
# --- Check 3: Zero ratio per end-use ---
zero_by_enduse = (
    df[df.is_country_group == False]
    .groupby("end_use")
    .agg(
        rows=("energy_tj", "size"),
        zeros=("energy_tj", lambda s: (s == 0).sum()),
        positives=("energy_tj", lambda s: (s > 0).sum()),
    )
    .assign(zero_pct=lambda x: (x["zeros"] / x["rows"] * 100).round(1))
    .sort_values("zero_pct", ascending=False)
)

zero_by_enduse

,rows,zeros,positives,zero_pct
end_use,,,,
Final consumption - other sectors - households - energy use - other end use,6270,5548,722,88.50
Final consumption - other sectors - households - energy use - cooking,6311,3547,2764,56.20
Final consumption - other sectors - households - energy use - water heating,6321,2316,4005,36.60
Final consumption - other sectors - households - energy use - space heating,6317,1649,4668,26.10
Final consumption - other sectors - households - energy use,6321,1439,4882,22.80
Final consumption - other sectors - households - energy use - space cooling,908,176,732,19.40
Final consumption - other sectors - households - energy use - lighting and electrical appliances,908,0,908,0.00


In [18]:
# --- Check 4: End-use × Fuel cell counts (which combos are empty?) ---
empty_combos = (
    combo_matrix
    .where(combo_matrix == 0)
    .stack()
    .reset_index()
    .drop(columns=0)
    .rename(columns={"end_use_code": "end_use", "fuel_code": "fuel"})
)

pd.DataFrame({
    "empty_combos_count": [len(empty_combos)],
    "examples": [empty_combos.head(10).to_dict(orient="records")],
}).T

,0
empty_combos_count,98
examples,"[{'end_use': 'FC_OTH_HH_E', 'fuel': 'E7000'}, ..."


In [19]:
# --- Check 5: Split feasibility — rows per period ---
split_check = (
    df[df.is_country_group == False]
    .assign(period_group=lambda d: np.where(
        d.year <= 2019, "train_2010_2019",
        np.where(d.year <= 2022, "val_2020_2022", "test_2023_2024")
    ))
    .groupby("period_group")
    .agg(rows=("energy_tj", "size"),
         positive_rows=("energy_tj", lambda s: (s > 0).sum()))
    .assign(positive_pct=lambda x: (x["positive_rows"] / x["rows"] * 100).round(1))
)

split_check

,rows,positive_rows,positive_pct
period_group,,,
test_2023_2024,5338,3080,57.70
train_2010_2019,19980,11011,55.10
val_2020_2022,8038,4590,57.10


In [20]:
# --- Check 6: Countries eligibility for forecasting ---
forecast_eligible = continuity.copy()
forecast_eligible["n_years"] = forecast_eligible.n_years
forecast_eligible["eligible_strict"] = (
    forecast_eligible.is_continuous & (forecast_eligible.n_years >= 10)
)
forecast_eligible["eligible_loose"] = forecast_eligible.is_continuous

pd.DataFrame({
    "criterion": [
        "continuous only",
        "continuous AND >=10 years",
        "continuous AND full 15 years",
    ],
    "countries": [
        forecast_eligible.eligible_loose.sum(),
        forecast_eligible.eligible_strict.sum(),
        (forecast_eligible.n_years == 15).sum(),
    ],
})

,criterion,countries
0,continuous only,38
1,continuous AND >=10 years,30
2,continuous AND full 15 years,17
